# Parquet & Efficient Data Formats

## Definition
Choosing the right file format has massive impact on storage costs, read speed, and ML pipeline efficiency.

### Format Comparison
| Format | Read | Write | Size | Schema | Best For |
|--------|------|-------|------|--------|---------|
| **CSV** | Slow | Slow | Large | No | Sharing, interoperability |
| **JSON** | Slow | Slow | Largest | No | APIs, config |
| **Pickle** | Fast | Fast | Medium | Python-only | Caching pandas objects |
| **Parquet** | Very Fast | Fast | Small | Yes | ML pipelines, columnar |
| **Feather** | Fastest | Fastest | Small | Yes | Quick pandas cache |
| **HDF5** | Fast | Fast | Small | Yes | Scientific data |

### Why Parquet for ML?
- **Columnar storage** — reads only needed columns
- **Built-in compression** (Snappy, GZIP, ZSTD)
- **Schema preserved** — dtypes survive (int8, float32, category, datetime)
- **Partitioned reads** — read only specific data partitions
- **Industry standard** — Spark, BigQuery, Athena, DuckDB all support it

In [ ]:
import pandas as pd
import numpy as np
import os
import time
import tempfile
import shutil

rng = np.random.default_rng(42)
n = 100_000

df = pd.DataFrame({
    'user_id':    rng.integers(0, 10000, n).astype('int32'),
    'age':        rng.integers(18, 65, n).astype('int8'),
    'income':     rng.normal(60000, 20000, n).astype('float32'),
    'score':      rng.uniform(0, 100, n).astype('float32'),
    'dept':       pd.Categorical(rng.choice(['Eng', 'Mkt', 'HR', 'Finance'], n)),
    'date':       pd.date_range('2022-01-01', periods=n, freq='min')[:n],
    'target':     rng.choice([0, 1], n, p=[0.8, 0.2]).astype('int8')
})

tmp = tempfile.mkdtemp()
print('DataFrame:', df.shape)
print(df.dtypes)
print(f'Memory: {df.memory_usage(deep=True).sum()/1024**2:.2f} MB')

In [ ]:
# Write and benchmark all formats
results = {}

# --- CSV ---
csv_path = os.path.join(tmp, 'data.csv')
t0 = time.time()
df.to_csv(csv_path, index=False)
write_ms = (time.time()-t0)*1000
t0 = time.time()
df_csv = pd.read_csv(csv_path)
read_ms = (time.time()-t0)*1000
results['CSV'] = {'write_ms': write_ms, 'read_ms': read_ms,
                  'size_mb': os.path.getsize(csv_path)/1024**2}

# --- Pickle ---
pkl_path = os.path.join(tmp, 'data.pkl')
t0 = time.time()
df.to_pickle(pkl_path)
write_ms = (time.time()-t0)*1000
t0 = time.time()
df_pkl = pd.read_pickle(pkl_path)
read_ms = (time.time()-t0)*1000
results['Pickle'] = {'write_ms': write_ms, 'read_ms': read_ms,
                     'size_mb': os.path.getsize(pkl_path)/1024**2}

print('CSV and Pickle benchmarked.')

In [ ]:
# --- Parquet ---
try:
    pq_path = os.path.join(tmp, 'data.parquet')
    t0 = time.time()
    df.to_parquet(pq_path, index=False, compression='snappy')
    write_ms = (time.time()-t0)*1000
    t0 = time.time()
    df_pq = pd.read_parquet(pq_path)
    read_ms = (time.time()-t0)*1000
    results['Parquet'] = {'write_ms': write_ms, 'read_ms': read_ms,
                          'size_mb': os.path.getsize(pq_path)/1024**2}
    print('Parquet OK. Dtypes preserved:')
    print(df_pq.dtypes)
except ImportError:
    print('pyarrow not installed: pip install pyarrow')

In [ ]:
# --- Feather ---
try:
    feather_path = os.path.join(tmp, 'data.feather')
    t0 = time.time()
    df.to_feather(feather_path)
    write_ms = (time.time()-t0)*1000
    t0 = time.time()
    df_feather = pd.read_feather(feather_path)
    read_ms = (time.time()-t0)*1000
    results['Feather'] = {'write_ms': write_ms, 'read_ms': read_ms,
                          'size_mb': os.path.getsize(feather_path)/1024**2}
    print('Feather OK.')
except Exception as e:
    print(f'Feather error: {e}')

In [ ]:
# Print benchmark comparison table
print(f"{'Format':10s}  {'Write(ms)':>10s}  {'Read(ms)':>10s}  {'Size(MB)':>10s}")
print('-' * 50)
for fmt, stats in results.items():
    print(f"{fmt:10s}  {stats['write_ms']:>10.1f}  "
          f"{stats['read_ms']:>10.1f}  {stats['size_mb']:>10.2f}")

# CSV baseline comparisons
csv_size = results['CSV']['size_mb']
print()
for fmt, stats in results.items():
    if fmt != 'CSV':
        reduction = (1 - stats['size_mb'] / csv_size) * 100
        print(f'{fmt} size reduction vs CSV: {reduction:.0f}%')

In [ ]:
# Parquet — selective column read (killer feature for ML)
try:
    t0 = time.time()
    df_all = pd.read_parquet(pq_path)
    t_all = time.time() - t0

    t0 = time.time()
    df_2col = pd.read_parquet(pq_path, columns=['income', 'target'])
    t_2col = time.time() - t0

    print(f'Full read ({df_all.shape[1]} cols):  {t_all*1000:.1f}ms')
    print(f'2-col read:                          {t_2col*1000:.1f}ms')
    print(f'Speedup: {t_all/max(t_2col, 0.001):.1f}x faster')
except Exception as e:
    print(e)

In [ ]:
# Partitioned Parquet (partition by category column)
try:
    part_path = os.path.join(tmp, 'partitioned')
    df.to_parquet(part_path, partition_cols=['dept'], index=False)

    print('Partition directory structure:')
    for root, dirs, files in os.walk(part_path):
        level = root.replace(part_path, '').count(os.sep)
        indent = '  ' * level
        folder = os.path.basename(root)
        print(f'{indent}{folder}/')
        for f in files[:1]:
            print(f'{indent}  {f}')

    # Read only one partition (Eng department)
    eng_part = os.path.join(part_path, 'dept=Eng')
    df_eng = pd.read_parquet(eng_part)
    print(f'\nEng partition only: {df_eng.shape[0]:,} rows')
except Exception as e:
    print(e)
finally:
    shutil.rmtree(tmp, ignore_errors=True)
    print('Temp files cleaned up.')

In [ ]:
# Summary: ML Data Format Recommendations
recommendations = {
    'Training/test sets': 'Parquet (dtype preservation, fast, compressed)',
    'Quick EDA cache':    'Feather (fastest read/write for pandas)',
    'Sklearn pipeline':   'Joblib or Pickle',
    'Sharing with others':'CSV (universal) or Parquet (modern)',
    'Very large data':    'Partitioned Parquet with Dask or Spark',
    'Config/metadata':    'JSON (human-readable)',
    'Avoid':              'CSV for >100MB numeric data in production'
}

print('=== ML DATA FORMAT RECOMMENDATIONS ===')
for use_case, rec in recommendations.items():
    print(f'  {use_case:30s}: {rec}')